# Smart Buildings: per-zone PI control and AHU cooling hold

**Task.** Control the HVAC of an unseen building every 5 minutes for 30 days
(8,640 steps). **Metric:** mean over scenarios of `abs(mean(reward))`, lower is
better, where reward weighs comfort 0.6, energy cost 0.2 and carbon 0.2.

**Per-zone PI control (main gain of this version).** Per-step logs showed
occupied zones swinging 2–4 K every 5–15 minutes: the organizer zone loop is
proportional only and opens the damper fully at 2 K of error, so a zone slightly
out of band gets a burst of air and overshoots through the whole band. Zones
averaged inside the 21–24 °C band yet were outside it 57–61% of work hours, and
comfort was 60% of the loss. A softer proportional loop stops the oscillation
but leaves zones sitting outside the band (no integral action).

Here every zone runs its own PI loop on (temperature − band centre) from one
hour before occupancy to the end of the work day (Kp 0.1, Ki
0.005 per 5-minute step, conditional integration against wind-up).
One signal drives the damper (|u|) and the reheat valve (−u); AHUs run through
those hours and zone reheat demand raises the hot-water temperature. Night
logic is unchanged. On full training episodes the comfort loss nearly
disappears (0.0006 mean on five scenarios with Kp 0.1, Ki 0.005).

**Four further changes on top of the per-zone PI loop** (this version's controller scores 0.01420 on the public leaderboard):

1. **Per-zone economic target.** Each zone keeps a one-day average of its own PI output; a zone that mostly needs cooling aims up to 0.6 K above the band centre, one that mostly needs heating aims up to 0.6 K below it (kept 0.3 K inside the band). Holding zones nearer the cheap edge of the comfort band saves energy.
2. **Supply temperature from the median zone demand.** An AHU chooses its supply-air temperature from the median of its zones' cooling demand rather than the mean, so a few outliers do not set it for all.
3. **Weekend heating setback.** On weekend days the heating setpoint is lowered by a further 6 K (nobody is in the building and comfort is not scored); weekday logic is unchanged.
4. **Minimum fan static pressure in work hours.** Every running air handler is held at its minimum static pressure during work hours instead of a pressure that follows the PI effort. Less airflow means less outside air to heat or cool, which is most of the energy; on the training scenarios this removed about 3% of the loss with no scenario worse.

Each change was screened on ten full 30-day training scenarios and then confirmed on the public leaderboard before being kept.


**Cooling-run hold.** The organizer PID controller stops an air handler (AHU)
the moment its zones are back inside the comfort band, and restarts it at the
next excursion: every AHU switched on or off 50–107 times a day. Keeping a unit
that was started **for cooling** running for at least 288 steps
(24 h) at its last commands removes most of the loss in
hot climates and leaves heating-dominated climates unchanged.

| Version | Controller | Public LB |
| --- | --- | --- |
| 3 | First baseline (mean-zone PID, sensor comfort bands, decoded weather) | 0.06439 |
| 4 | + cooling-run hold, 6 steps | 0.05882 |
| 5 | + cooling-run hold, 288 steps + night cooling ceiling | 0.05195 |
| this | v5 + per-zone PI + per-zone economic targets + median-demand supply temperature + weekend heating setback + minimum fan pressure in work hours (Kp 0.1, Ki 0.005) | 0.01420 (identical private probe) |

Sources: [competition](https://www.kaggle.com/competitions/neurips-2026-smart-buildings),
[starter kit](https://github.com/smart-buildings-2026/starter-kit). The base
controller is the organizers' `PIDWaterResetPolicy` (Apache-2.0). The EDA reads
only published training scenarios and is skipped during the hidden rerun.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path('/kaggle/input/competitions/neurips-2026-smart-buildings'),
              Path('/kaggle/input/neurips-2026-smart-buildings')]
if os.getenv('SMART_BUILDINGS_DATA_ROOT'):
    candidates.insert(0, Path(os.environ['SMART_BUILDINGS_DATA_ROOT']))
DATA_ROOT = next((p for p in candidates if (p / 'smart_buildings').exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Attach competition data or set SMART_BUILDINGS_DATA_ROOT')
sys.path.insert(0, str(DATA_ROOT))
sys.path.insert(0, str(DATA_ROOT / 'kaggle_evaluation'))
IS_RERUN = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is not None
print('Competition data located; hidden rerun:', IS_RERUN)


## Exploratory data analysis

30 training scenarios: 6 buildings × 5 climates (Atlanta, Denver, Fairbanks,
Phoenix, Singapore); 30–85 zones and 3–9 AHUs per building. Below: building
sizes, observation and action dimensions, initial sensor gaps, and the actual
30-day weather windows the simulator uses. A zero continuous action is the
middle of the range (e.g. a half-open damper), not "off". AHUs can only cool;
heat comes from zone hot-water reheat, which needs airflow to deliver it.


In [ ]:
"""EDA of released training data; never opens evaluation_scenarios."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml


def analyze_data(data_root, output_root):
    data_root = Path(data_root).resolve()
    output_root = Path(output_root).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    from smart_buildings.evaluate import load_hybrid_environment, create_building_meta_data
    previous = Path.cwd()
    buildings, weather_rows = [], []
    frames = {}
    try:
        os.chdir(data_root)
        scenario_dirs = sorted((data_root / "smart_buildings/training_scenarios").glob("*/*"))
        seen = set()
        for scenario in scenario_dirs:
            if not (scenario / "config.gin").exists():
                continue
            cfg = yaml.safe_load((scenario / "config.yaml").read_text())
            if scenario.parent.name not in seen:
                seen.add(scenario.parent.name)
                env = load_hybrid_environment(scenario / "config.gin")
                # Competition input is read-only on Kaggle, including metric paths.
                env.metrics_path = None
                meta = create_building_meta_data(env, scenario)
                obs, _ = env.reset()
                assert np.isfinite(obs).all(), "Non-finite initial observation"
                names = meta["observation_field_names"]
                zone_fields = [n for n in names if n.endswith("_zone_air_temperature_sensor")]
                v, k = len(meta["zone_action_indices"]), len(meta["temp_indices"])
                assert meta["n_continuous_action"] == 3*k + 2 + 2*v
                assert len(zone_fields) == v
                floor = np.load(scenario / "floor_plan.npy")
                buildings.append(dict(
                    building=scenario.parent.name, floors=meta["num_floors"],
                    zones=v, ahus=k, observation_dim=len(names),
                    continuous_dim=meta["n_continuous_action"],
                    discrete_dim=meta["n_discrete_action"],
                    grid_cells=int(floor.size), initial_missing=int(np.isnan(obs).sum()),
                    exterior_conductivity=cfg["materials"]["exterior_cv_conductivity"],
                    exterior_density=cfg["materials"]["exterior_cv_density"],
                    interior_mass_density=cfg["materials"]["interior_mass_cv_density"],
                ))
                if len(buildings) == 1:
                    pd.DataFrame(dict(field=names, initial_observation=obs)).to_csv(
                        output_root / "observation_example.csv", index=False)
                    for key in ("observation_normalization_metadata", "action_normalization_metadata"):
                        import json
                        (output_root / f"{key}.json").write_text(json.dumps(meta[key], indent=2))
                    fig, ax = plt.subplots(figsize=(9, 5))
                    ax.imshow(floor, cmap="viridis", interpolation="nearest")
                    ax.set(title=f"{scenario.parent.name}: simulator floor grid", xlabel="grid column", ylabel="grid row")
                    fig.tight_layout(); fig.savefig(output_root / "floor_plan.png", dpi=150); plt.close(fig)
            csv = data_root / "smart_buildings/weather" / cfg["weather"]["weather_csv"]
            if csv not in frames:
                frame = pd.read_csv(csv)
                frame["timestamp"] = pd.to_datetime(frame["Time"], utc=True)
                frames[csv] = frame
            frame = frames[csv]
            # Gin's actual simulator clock takes the weather override.
            start = pd.Timestamp(cfg["weather"]["start_timestamp"], tz="UTC")
            days = int(cfg["simulation"]["num_days_in_episode"])
            window = frame[(frame.timestamp >= start) & (frame.timestamp < start + pd.Timedelta(days=days))]
            temp = (window.TempF - 32) * (5/9)
            weather_rows.append(dict(
                building=scenario.parent.name, weather=scenario.name,
                station=scenario.name.rsplit("_", 1)[0], start=str(start),
                hours=len(window), missing_temperatures=int(temp.isna().sum()),
                duplicate_timestamps=int(window.timestamp.duplicated().sum()),
                min_C=float(temp.min()), mean_C=float(temp.mean()), max_C=float(temp.max()),
                hours_below_minus6_8_pct=float(100*(temp < -6.8).mean()),
            ))
    finally:
        os.chdir(previous)
    building_df, weather_df = pd.DataFrame(buildings), pd.DataFrame(weather_rows)
    building_df.to_csv(output_root / "buildings.csv", index=False)
    weather_df.to_csv(output_root / "weather_strips.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    building_df.plot.bar(x="building", y=["zones", "ahus"], ax=axes[0], rot=25)
    axes[0].set_title("Variable building and action size")
    grouped = weather_df.groupby("station").agg(min_C=("min_C", "min"), mean_C=("mean_C", "mean"), max_C=("max_C", "max"))
    x = np.arange(len(grouped))
    axes[1].vlines(x, grouped.min_C, grouped.max_C, linewidth=5, color="#3b82f6")
    axes[1].scatter(x, grouped.mean_C, color="#f97316", label="mean across strips")
    axes[1].set(xticks=x, xticklabels=grouped.index, ylabel="outside temperature (C)", title="30-day training weather windows")
    axes[1].legend(); fig.tight_layout(); fig.savefig(output_root / "eda.png", dpi=150); plt.close(fig)
    print(building_df.to_string(index=False))
    print(grouped.round(2).to_string())
    print(f"Scenarios: {len(weather_df)}, initial observation NaNs: {building_df.initial_missing.sum()}")
    return building_df, weather_df


In [ ]:
if not IS_RERUN:
    buildings, weather = analyze_data(DATA_ROOT, Path.cwd() / 'eda')
    from IPython.display import display, Image
    display(buildings)
    display(weather)
    display(Image(filename='eda/eda.png'))
    display(Image(filename='eda/floor_plan.png'))


## How the improvement was found

All numbers are full 30-day episodes run on Kaggle CPU with the official
simulator. Lower is better.

1. **The comparison is noisy.** Two runs that differ by a hundredth of a pascal
   in one command track each other for days, then jump ~10 K apart when a
   switching decision flips. A *null group* (the baseline plus three runs with
   such negligible nudges) moves the episode score by 1–6%. A candidate is
   judged by `z = (score − null mean) / null sd`; |z| < 2 is noise.
2. **Rejected:** flow-preserving fan pressure reset (inside the noise),
   colder supply air (−74% on Singapore), delayed AHU starts (worse comfort).
3. **Minimum run time** helped hot climates but hurt Fairbanks, where it
   lengthened heating bursts. Holding **only cooling runs** kept the hot-climate
   gain and removed the cold-climate loss, and the gain grew with hold length.
4. **Zone oscillation.** After the hold, comfort was still 60% of the loss:
   zones swung 2–4 K every few steps. Softer proportional gains removed the
   swings but left a steady offset; per-zone PI to the band centre fixed both.
   On five full scenarios (two from a building not used for tuning) the PI
   controller scored 0.0204 on average against 0.0483 for the hold alone.

Pilot screening (building_train_8, three climates). `z_by_scenario` order:
Fairbanks / Phoenix / Singapore; negative is better.


In [ ]:
pilot = pd.DataFrame(json.loads('[{"policy": "min_on_cool_288", "building_train_1/atlanta_2": 0.021844, "building_train_1/denver_7": 0.024986, "building_train_1/fairbanks_1": 0.214082, "building_train_1/phoenix_5": 0.059433, "building_train_1/singapore_8": 0.024687, "building_train_2/atlanta_3": 0.016204, "building_train_2/denver_8": 0.034157, "building_train_2/fairbanks_2": 0.070825, "building_train_2/phoenix_6": 0.034516, "building_train_2/singapore_1": 0.01804, "building_train_4/atlanta_5": 0.024324, "building_train_4/denver_2": 0.052013, "building_train_4/fairbanks_4": 0.024768, "building_train_4/phoenix_8": 0.022073, "building_train_4/singapore_3": 0.025551, "building_train_5/atlanta_6": 0.014049, "building_train_5/denver_3": 0.037197, "building_train_5/fairbanks_5": 0.012584, "building_train_5/phoenix_1": 0.026367, "building_train_5/singapore_4": 0.024586, "building_train_7/atlanta_8": 0.046477, "building_train_7/denver_5": 0.018875, "building_train_7/fairbanks_7": 0.053886, "building_train_7/phoenix_3": 0.02568, "building_train_7/singapore_6": 0.0215, "building_train_8/atlanta_1": 0.0318, "building_train_8/denver_6": 0.01564, "building_train_8/fairbanks_8": 0.077077, "building_train_8/phoenix_4": 0.044971, "building_train_8/singapore_7": 0.0191, "mean_improvement_pct": 45.280909, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 0.0 / -27.4 / -24.2"}, {"policy": "min_on_cool_96", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077077, "building_train_8/phoenix_4": 0.055115, "building_train_8/singapore_7": 0.019998, "mean_improvement_pct": 42.639779, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 0.0 / -25.1 / -23.5"}, {"policy": "min_on_cool_48", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077077, "building_train_8/phoenix_4": 0.058816, "building_train_8/singapore_7": 0.020033, "mean_improvement_pct": 41.863527, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 0.0 / -24.2 / -23.5"}, {"policy": "min_on_cool_24", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077077, "building_train_8/phoenix_4": 0.078474, "building_train_8/singapore_7": 0.02173, "mean_improvement_pct": 36.773736, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 0.0 / -19.7 / -22.3"}, {"policy": "min_on_cool_12", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077886, "building_train_8/phoenix_4": 0.098678, "building_train_8/singapore_7": 0.021685, "mean_improvement_pct": 32.338253, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 2.2 / -15.0 / -22.3"}, {"policy": "min_on_cool_6", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077316, "building_train_8/phoenix_4": 0.102337, "building_train_8/singapore_7": 0.023848, "mean_improvement_pct": 30.4552, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 0.7 / -14.2 / -20.7"}, {"policy": "min_on_cool_3", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077297, "building_train_8/phoenix_4": 0.128013, "building_train_8/singapore_7": 0.030711, "mean_improvement_pct": 20.842412, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 0.6 / -8.2 / -15.7"}, {"policy": "min_on_6", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.094148, "building_train_8/phoenix_4": 0.105296, "building_train_8/singapore_7": 0.03669, "mean_improvement_pct": 14.355608, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 45.2 / -13.5 / -11.3"}, {"policy": "min_on_3", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.087441, "building_train_8/phoenix_4": 0.123693, "building_train_8/singapore_7": 0.035456, "mean_improvement_pct": 14.298793, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 27.5 / -9.2 / -12.2"}, {"policy": "null_2", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077393, "building_train_8/phoenix_4": 0.159066, "building_train_8/singapore_7": 0.052083, "mean_improvement_pct": 0.801494, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 0.9 / -1.1 / -0.0"}, {"policy": "null_1", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.076552, "building_train_8/phoenix_4": 0.162694, "building_train_8/singapore_7": 0.052084, "mean_improvement_pct": 0.425973, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / -1.4 / -0.2 / -0.0"}, {"policy": "null_3", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077011, "building_train_8/phoenix_4": 0.169492, "building_train_8/singapore_7": 0.05044, "mean_improvement_pct": -0.104937, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / -0.1 / 1.3 / -1.2"}, {"policy": "transfer", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077297, "building_train_8/phoenix_4": 0.163381, "building_train_8/singapore_7": 0.053782, "mean_improvement_pct": -1.122531, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 0.6 / -0.1 / 1.2"}, {"policy": "pressure_reset", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077847, "building_train_8/phoenix_4": 0.166517, "building_train_8/singapore_7": 0.052698, "mean_improvement_pct": -1.305578, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 2.1 / 0.7 / 0.4"}, {"policy": "min_on_2", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.077797, "building_train_8/phoenix_4": 0.157236, "building_train_8/singapore_7": 0.056332, "mean_improvement_pct": -1.718751, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 1.9 / -1.5 / 3.1"}, {"policy": "start_02", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.082354, "building_train_8/phoenix_4": 0.147884, "building_train_8/singapore_7": 0.063, "mean_improvement_pct": -6.051391, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 14.0 / -3.6 / 8.0"}, {"policy": "start_05", "building_train_1/atlanta_2": NaN, "building_train_1/denver_7": NaN, "building_train_1/fairbanks_1": NaN, "building_train_1/phoenix_5": NaN, "building_train_1/singapore_8": NaN, "building_train_2/atlanta_3": NaN, "building_train_2/denver_8": NaN, "building_train_2/fairbanks_2": NaN, "building_train_2/phoenix_6": NaN, "building_train_2/singapore_1": NaN, "building_train_4/atlanta_5": NaN, "building_train_4/denver_2": NaN, "building_train_4/fairbanks_4": NaN, "building_train_4/phoenix_8": NaN, "building_train_4/singapore_3": NaN, "building_train_5/atlanta_6": NaN, "building_train_5/denver_3": NaN, "building_train_5/fairbanks_5": NaN, "building_train_5/phoenix_1": NaN, "building_train_5/singapore_4": NaN, "building_train_7/atlanta_8": NaN, "building_train_7/denver_5": NaN, "building_train_7/fairbanks_7": NaN, "building_train_7/phoenix_3": NaN, "building_train_7/singapore_6": NaN, "building_train_8/atlanta_1": NaN, "building_train_8/denver_6": NaN, "building_train_8/fairbanks_8": 0.089897, "building_train_8/phoenix_4": 0.184582, "building_train_8/singapore_7": 0.112904, "mean_improvement_pct": -48.718099, "z_by_scenario": "nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / nan / 34.0 / 4.8 / 44.6"}]'))
if not IS_RERUN:
    display(pilot)


## Check on all 30 training scenarios

The hold length was tuned on building_train_8, so the other five buildings
(25 scenarios) are the out-of-sample check. Below: per-scenario scores of the
baseline (`transfer`) and the hold (`min_on_cool_288`), and the mean by climate.
These are training scenarios; they do not estimate the hidden leaderboard.


In [ ]:
full = pd.DataFrame(json.loads('[{"scenario": "building_train_1/atlanta_2", "policy": "min_on_cool_6", "steps": 8640, "score": 0.023757, "comfort_loss": 0.019626, "energy_loss": 0.002198, "carbon_loss": 0.001933}, {"scenario": "building_train_1/atlanta_2", "policy": "transfer", "steps": 8640, "score": 0.024181, "comfort_loss": 0.019969, "energy_loss": 0.002269, "carbon_loss": 0.001942}, {"scenario": "building_train_1/denver_7", "policy": "min_on_cool_6", "steps": 8640, "score": 0.029103, "comfort_loss": 0.028222, "energy_loss": 0.000431, "carbon_loss": 0.00045}, {"scenario": "building_train_1/denver_7", "policy": "transfer", "steps": 8640, "score": 0.029976, "comfort_loss": 0.029086, "energy_loss": 0.000445, "carbon_loss": 0.000445}, {"scenario": "building_train_1/fairbanks_1", "policy": "min_on_cool_6", "steps": 8640, "score": 0.214244, "comfort_loss": 0.070473, "energy_loss": 0.055038, "carbon_loss": 0.088733}, {"scenario": "building_train_1/fairbanks_1", "policy": "transfer", "steps": 8640, "score": 0.212925, "comfort_loss": 0.069706, "energy_loss": 0.054741, "carbon_loss": 0.088478}, {"scenario": "building_train_1/phoenix_5", "policy": "min_on_cool_6", "steps": 8640, "score": 0.104265, "comfort_loss": 0.052861, "energy_loss": 0.037869, "carbon_loss": 0.013536}, {"scenario": "building_train_1/phoenix_5", "policy": "transfer", "steps": 8640, "score": 0.136072, "comfort_loss": 0.074278, "energy_loss": 0.045083, "carbon_loss": 0.016711}, {"scenario": "building_train_1/singapore_8", "policy": "min_on_cool_6", "steps": 8640, "score": 0.025284, "comfort_loss": 0.015158, "energy_loss": 0.008101, "carbon_loss": 0.002026}, {"scenario": "building_train_1/singapore_8", "policy": "transfer", "steps": 8640, "score": 0.028131, "comfort_loss": 0.017798, "energy_loss": 0.008262, "carbon_loss": 0.002071}, {"scenario": "building_train_2/atlanta_3", "policy": "min_on_cool_6", "steps": 8640, "score": 0.019147, "comfort_loss": 0.014722, "energy_loss": 0.003566, "carbon_loss": 0.000859}, {"scenario": "building_train_2/atlanta_3", "policy": "transfer", "steps": 8640, "score": 0.020152, "comfort_loss": 0.015595, "energy_loss": 0.003674, "carbon_loss": 0.000884}, {"scenario": "building_train_2/denver_8", "policy": "min_on_cool_6", "steps": 8640, "score": 0.038612, "comfort_loss": 0.032884, "energy_loss": 0.002491, "carbon_loss": 0.003237}, {"scenario": "building_train_2/denver_8", "policy": "transfer", "steps": 8640, "score": 0.039837, "comfort_loss": 0.034068, "energy_loss": 0.002537, "carbon_loss": 0.003232}, {"scenario": "building_train_2/fairbanks_2", "policy": "min_on_cool_6", "steps": 8640, "score": 0.077485, "comfort_loss": 0.058128, "energy_loss": 0.007402, "carbon_loss": 0.011955}, {"scenario": "building_train_2/fairbanks_2", "policy": "transfer", "steps": 8640, "score": 0.079776, "comfort_loss": 0.060406, "energy_loss": 0.007428, "carbon_loss": 0.011942}, {"scenario": "building_train_2/phoenix_6", "policy": "min_on_cool_6", "steps": 8640, "score": 0.053152, "comfort_loss": 0.029955, "energy_loss": 0.017501, "carbon_loss": 0.005696}, {"scenario": "building_train_2/phoenix_6", "policy": "transfer", "steps": 8640, "score": 0.070571, "comfort_loss": 0.040747, "energy_loss": 0.022018, "carbon_loss": 0.007807}, {"scenario": "building_train_2/singapore_1", "policy": "min_on_cool_6", "steps": 8640, "score": 0.021724, "comfort_loss": 0.01415, "energy_loss": 0.00608, "carbon_loss": 0.001494}, {"scenario": "building_train_2/singapore_1", "policy": "transfer", "steps": 8640, "score": 0.026442, "comfort_loss": 0.017387, "energy_loss": 0.007053, "carbon_loss": 0.002002}, {"scenario": "building_train_4/atlanta_5", "policy": "min_on_cool_6", "steps": 8640, "score": 0.026179, "comfort_loss": 0.016057, "energy_loss": 0.008194, "carbon_loss": 0.001928}, {"scenario": "building_train_4/atlanta_5", "policy": "transfer", "steps": 8640, "score": 0.028679, "comfort_loss": 0.018379, "energy_loss": 0.008352, "carbon_loss": 0.001948}, {"scenario": "building_train_4/denver_2", "policy": "min_on_cool_6", "steps": 8640, "score": 0.054021, "comfort_loss": 0.04858, "energy_loss": 0.002072, "carbon_loss": 0.00337}, {"scenario": "building_train_4/denver_2", "policy": "transfer", "steps": 8640, "score": 0.054502, "comfort_loss": 0.049156, "energy_loss": 0.002036, "carbon_loss": 0.00331}, {"scenario": "building_train_4/fairbanks_4", "policy": "min_on_cool_6", "steps": 8640, "score": 0.030803, "comfort_loss": 0.029361, "energy_loss": 0.000969, "carbon_loss": 0.000473}, {"scenario": "building_train_4/fairbanks_4", "policy": "transfer", "steps": 8640, "score": 0.031825, "comfort_loss": 0.030207, "energy_loss": 0.00111, "carbon_loss": 0.000508}, {"scenario": "building_train_4/phoenix_8", "policy": "min_on_cool_6", "steps": 8640, "score": 0.028171, "comfort_loss": 0.026614, "energy_loss": 0.001167, "carbon_loss": 0.00039}, {"scenario": "building_train_4/phoenix_8", "policy": "transfer", "steps": 8640, "score": 0.029013, "comfort_loss": 0.027287, "energy_loss": 0.001302, "carbon_loss": 0.000424}, {"scenario": "building_train_4/singapore_3", "policy": "min_on_cool_6", "steps": 8640, "score": 0.026857, "comfort_loss": 0.015657, "energy_loss": 0.009063, "carbon_loss": 0.002137}, {"scenario": "building_train_4/singapore_3", "policy": "transfer", "steps": 8640, "score": 0.035134, "comfort_loss": 0.021013, "energy_loss": 0.0109, "carbon_loss": 0.003221}, {"scenario": "building_train_5/atlanta_6", "policy": "min_on_cool_6", "steps": 8640, "score": 0.021838, "comfort_loss": 0.014134, "energy_loss": 0.00608, "carbon_loss": 0.001624}, {"scenario": "building_train_5/atlanta_6", "policy": "transfer", "steps": 8640, "score": 0.03416, "comfort_loss": 0.022188, "energy_loss": 0.008863, "carbon_loss": 0.003109}, {"scenario": "building_train_5/denver_3", "policy": "min_on_cool_6", "steps": 8640, "score": 0.04218, "comfort_loss": 0.037471, "energy_loss": 0.001789, "carbon_loss": 0.00292}, {"scenario": "building_train_5/denver_3", "policy": "transfer", "steps": 8640, "score": 0.043519, "comfort_loss": 0.038822, "energy_loss": 0.001793, "carbon_loss": 0.002905}, {"scenario": "building_train_5/fairbanks_5", "policy": "min_on_cool_6", "steps": 8640, "score": 0.01637, "comfort_loss": 0.014359, "energy_loss": 0.001613, "carbon_loss": 0.000399}, {"scenario": "building_train_5/fairbanks_5", "policy": "transfer", "steps": 8640, "score": 0.016909, "comfort_loss": 0.01477, "energy_loss": 0.001716, "carbon_loss": 0.000423}, {"scenario": "building_train_5/phoenix_1", "policy": "min_on_cool_6", "steps": 8640, "score": 0.032137, "comfort_loss": 0.029887, "energy_loss": 0.001003, "carbon_loss": 0.001247}, {"scenario": "building_train_5/phoenix_1", "policy": "transfer", "steps": 8640, "score": 0.033239, "comfort_loss": 0.030978, "energy_loss": 0.001016, "carbon_loss": 0.001246}, {"scenario": "building_train_5/singapore_4", "policy": "min_on_cool_6", "steps": 8640, "score": 0.078257, "comfort_loss": 0.044401, "energy_loss": 0.023017, "carbon_loss": 0.010839}, {"scenario": "building_train_5/singapore_4", "policy": "transfer", "steps": 8640, "score": 0.107665, "comfort_loss": 0.061349, "energy_loss": 0.030598, "carbon_loss": 0.015718}, {"scenario": "building_train_7/atlanta_8", "policy": "min_on_cool_6", "steps": 8640, "score": 0.048211, "comfort_loss": 0.039145, "energy_loss": 0.003381, "carbon_loss": 0.005684}, {"scenario": "building_train_7/atlanta_8", "policy": "transfer", "steps": 8640, "score": 0.048465, "comfort_loss": 0.039426, "energy_loss": 0.003368, "carbon_loss": 0.005671}, {"scenario": "building_train_7/denver_5", "policy": "min_on_cool_6", "steps": 8640, "score": 0.029159, "comfort_loss": 0.018946, "energy_loss": 0.007654, "carbon_loss": 0.00256}, {"scenario": "building_train_7/denver_5", "policy": "transfer", "steps": 8640, "score": 0.036026, "comfort_loss": 0.023814, "energy_loss": 0.009045, "carbon_loss": 0.003166}, {"scenario": "building_train_7/fairbanks_7", "policy": "min_on_cool_6", "steps": 8640, "score": 0.053854, "comfort_loss": 0.042241, "energy_loss": 0.004153, "carbon_loss": 0.00746}, {"scenario": "building_train_7/fairbanks_7", "policy": "transfer", "steps": 8640, "score": 0.053778, "comfort_loss": 0.042199, "energy_loss": 0.004147, "carbon_loss": 0.007433}, {"scenario": "building_train_7/phoenix_3", "policy": "min_on_cool_6", "steps": 8640, "score": 0.036567, "comfort_loss": 0.022531, "energy_loss": 0.010612, "carbon_loss": 0.003424}, {"scenario": "building_train_7/phoenix_3", "policy": "transfer", "steps": 8640, "score": 0.044925, "comfort_loss": 0.028412, "energy_loss": 0.012163, "carbon_loss": 0.00435}, {"scenario": "building_train_7/singapore_6", "policy": "min_on_cool_6", "steps": 8640, "score": 0.024553, "comfort_loss": 0.014502, "energy_loss": 0.008049, "carbon_loss": 0.002002}, {"scenario": "building_train_7/singapore_6", "policy": "transfer", "steps": 8640, "score": 0.034823, "comfort_loss": 0.021331, "energy_loss": 0.010233, "carbon_loss": 0.003259}, {"scenario": "building_train_8/atlanta_1", "policy": "min_on_cool_6", "steps": 8640, "score": 0.032359, "comfort_loss": 0.024291, "energy_loss": 0.003056, "carbon_loss": 0.005012}, {"scenario": "building_train_8/atlanta_1", "policy": "transfer", "steps": 8640, "score": 0.032533, "comfort_loss": 0.024467, "energy_loss": 0.003069, "carbon_loss": 0.004997}, {"scenario": "building_train_8/denver_6", "policy": "min_on_cool_6", "steps": 8640, "score": 0.019112, "comfort_loss": 0.014077, "energy_loss": 0.004056, "carbon_loss": 0.000978}, {"scenario": "building_train_8/denver_6", "policy": "transfer", "steps": 8640, "score": 0.020257, "comfort_loss": 0.015097, "energy_loss": 0.004163, "carbon_loss": 0.000997}, {"scenario": "building_train_8/fairbanks_8", "policy": "min_on_cool_6", "steps": 8640, "score": 0.077316, "comfort_loss": 0.031317, "energy_loss": 0.016066, "carbon_loss": 0.029934}, {"scenario": "building_train_8/fairbanks_8", "policy": "transfer", "steps": 8640, "score": 0.077297, "comfort_loss": 0.031107, "energy_loss": 0.016144, "carbon_loss": 0.030046}, {"scenario": "building_train_8/phoenix_4", "policy": "min_on_cool_6", "steps": 8640, "score": 0.102337, "comfort_loss": 0.05544, "energy_loss": 0.03332, "carbon_loss": 0.013577}, {"scenario": "building_train_8/phoenix_4", "policy": "transfer", "steps": 8640, "score": 0.163381, "comfort_loss": 0.09317, "energy_loss": 0.048682, "carbon_loss": 0.021529}, {"scenario": "building_train_8/singapore_7", "policy": "min_on_cool_6", "steps": 8640, "score": 0.023848, "comfort_loss": 0.01431, "energy_loss": 0.007552, "carbon_loss": 0.001986}, {"scenario": "building_train_8/singapore_7", "policy": "transfer", "steps": 8640, "score": 0.053782, "comfort_loss": 0.032004, "energy_loss": 0.015029, "carbon_loss": 0.006748}, {"scenario": "building_train_1/atlanta_2", "policy": "min_on_cool_288", "steps": 8640, "score": 0.021844, "comfort_loss": 0.01768, "energy_loss": 0.002205, "carbon_loss": 0.001959}, {"scenario": "building_train_1/denver_7", "policy": "min_on_cool_288", "steps": 8640, "score": 0.024986, "comfort_loss": 0.023832, "energy_loss": 0.000537, "carbon_loss": 0.000616}, {"scenario": "building_train_1/fairbanks_1", "policy": "min_on_cool_288", "steps": 8640, "score": 0.214082, "comfort_loss": 0.070397, "energy_loss": 0.055012, "carbon_loss": 0.088673}, {"scenario": "building_train_1/phoenix_5", "policy": "min_on_cool_288", "steps": 8640, "score": 0.059433, "comfort_loss": 0.025426, "energy_loss": 0.026264, "carbon_loss": 0.007743}, {"scenario": "building_train_1/singapore_8", "policy": "min_on_cool_288", "steps": 8640, "score": 0.024687, "comfort_loss": 0.014527, "energy_loss": 0.008139, "carbon_loss": 0.002021}, {"scenario": "building_train_2/atlanta_3", "policy": "min_on_cool_288", "steps": 8640, "score": 0.016204, "comfort_loss": 0.011179, "energy_loss": 0.00399, "carbon_loss": 0.001036}, {"scenario": "building_train_2/denver_8", "policy": "min_on_cool_288", "steps": 8640, "score": 0.034157, "comfort_loss": 0.027327, "energy_loss": 0.002823, "carbon_loss": 0.004006}, {"scenario": "building_train_2/fairbanks_2", "policy": "min_on_cool_288", "steps": 8640, "score": 0.070825, "comfort_loss": 0.048753, "energy_loss": 0.008457, "carbon_loss": 0.013616}, {"scenario": "building_train_2/phoenix_6", "policy": "min_on_cool_288", "steps": 8640, "score": 0.034516, "comfort_loss": 0.016715, "energy_loss": 0.013898, "carbon_loss": 0.003904}, {"scenario": "building_train_2/singapore_1", "policy": "min_on_cool_288", "steps": 8640, "score": 0.01804, "comfort_loss": 0.009663, "energy_loss": 0.006722, "carbon_loss": 0.001655}, {"scenario": "building_train_4/atlanta_5", "policy": "min_on_cool_288", "steps": 8640, "score": 0.024324, "comfort_loss": 0.014176, "energy_loss": 0.008235, "carbon_loss": 0.001913}, {"scenario": "building_train_4/denver_2", "policy": "min_on_cool_288", "steps": 8640, "score": 0.052013, "comfort_loss": 0.046023, "energy_loss": 0.00225, "carbon_loss": 0.003739}, {"scenario": "building_train_4/fairbanks_4", "policy": "min_on_cool_288", "steps": 8640, "score": 0.024768, "comfort_loss": 0.023271, "energy_loss": 0.000926, "carbon_loss": 0.000571}, {"scenario": "building_train_4/phoenix_8", "policy": "min_on_cool_288", "steps": 8640, "score": 0.022073, "comfort_loss": 0.020401, "energy_loss": 0.001196, "carbon_loss": 0.000476}, {"scenario": "building_train_4/singapore_3", "policy": "min_on_cool_288", "steps": 8640, "score": 0.025551, "comfort_loss": 0.014363, "energy_loss": 0.009088, "carbon_loss": 0.002101}, {"scenario": "building_train_5/atlanta_6", "policy": "min_on_cool_288", "steps": 8640, "score": 0.014049, "comfort_loss": 0.006861, "energy_loss": 0.005826, "carbon_loss": 0.001363}, {"scenario": "building_train_5/denver_3", "policy": "min_on_cool_288", "steps": 8640, "score": 0.037197, "comfort_loss": 0.03204, "energy_loss": 0.001973, "carbon_loss": 0.003183}, {"scenario": "building_train_5/fairbanks_5", "policy": "min_on_cool_288", "steps": 8640, "score": 0.012584, "comfort_loss": 0.010512, "energy_loss": 0.001652, "carbon_loss": 0.000421}, {"scenario": "building_train_5/phoenix_1", "policy": "min_on_cool_288", "steps": 8640, "score": 0.026367, "comfort_loss": 0.02369, "energy_loss": 0.001181, "carbon_loss": 0.001495}, {"scenario": "building_train_5/singapore_4", "policy": "min_on_cool_288", "steps": 8640, "score": 0.024586, "comfort_loss": 0.011622, "energy_loss": 0.010014, "carbon_loss": 0.00295}, {"scenario": "building_train_7/atlanta_8", "policy": "min_on_cool_288", "steps": 8640, "score": 0.046477, "comfort_loss": 0.037122, "energy_loss": 0.003493, "carbon_loss": 0.005863}, {"scenario": "building_train_7/denver_5", "policy": "min_on_cool_288", "steps": 8640, "score": 0.018875, "comfort_loss": 0.010676, "energy_loss": 0.006456, "carbon_loss": 0.001743}, {"scenario": "building_train_7/fairbanks_7", "policy": "min_on_cool_288", "steps": 8640, "score": 0.053886, "comfort_loss": 0.041918, "energy_loss": 0.004285, "carbon_loss": 0.007683}, {"scenario": "building_train_7/phoenix_3", "policy": "min_on_cool_288", "steps": 8640, "score": 0.02568, "comfort_loss": 0.013867, "energy_loss": 0.009322, "carbon_loss": 0.002492}, {"scenario": "building_train_7/singapore_6", "policy": "min_on_cool_288", "steps": 8640, "score": 0.0215, "comfort_loss": 0.010992, "energy_loss": 0.008487, "carbon_loss": 0.002021}, {"scenario": "building_train_8/atlanta_1", "policy": "min_on_cool_288", "steps": 8640, "score": 0.0318, "comfort_loss": 0.023459, "energy_loss": 0.003166, "carbon_loss": 0.005175}, {"scenario": "building_train_8/denver_6", "policy": "min_on_cool_288", "steps": 8640, "score": 0.01564, "comfort_loss": 0.009264, "energy_loss": 0.004996, "carbon_loss": 0.001379}, {"scenario": "building_train_8/fairbanks_8", "policy": "min_on_cool_288", "steps": 8640, "score": 0.077077, "comfort_loss": 0.031436, "energy_loss": 0.015926, "carbon_loss": 0.029716}, {"scenario": "building_train_8/phoenix_4", "policy": "min_on_cool_288", "steps": 8640, "score": 0.044971, "comfort_loss": 0.018981, "energy_loss": 0.019934, "carbon_loss": 0.006056}, {"scenario": "building_train_8/singapore_7", "policy": "min_on_cool_288", "steps": 8640, "score": 0.0191, "comfort_loss": 0.00903, "energy_loss": 0.008106, "carbon_loss": 0.001963}]'))
CANDIDATE = 'min_on_cool_288'
if not IS_RERUN and len(full):
    scores = full.pivot(index='scenario', columns='policy', values='score')
    cols = [c for c in ('transfer', CANDIDATE) if c in scores]
    scores = scores[cols]
    scores['pilot_building'] = scores.index.str.startswith('building_train_8/')
    if len(cols) == 2:
        scores['change_pct'] = 100 * (scores[CANDIDATE] - scores['transfer']) / scores['transfer']
    display(scores.round(4))
    climate = scores[cols].groupby(scores.index.str.split('/').str[1].str.replace(r'_\d+$', '', regex=True)).mean()
    display(climate.round(4))
    summary = pd.DataFrame({
        'all 30': scores[cols].mean(),
        'non-pilot 25': scores.loc[~scores.pilot_building, cols].mean()})
    display(summary.round(5))
    ax = climate.plot.bar(figsize=(9, 4), rot=0)
    ax.set(title='Mean episode score by climate (lower is better)', ylabel='Score', xlabel='Climate')
    plt.tight_layout(); plt.show()


## Controller

`BaselinePolicy` is the first submitted controller: the organizers' PID with
per-zone comfort bands read from sensors and outside temperature decoded from
its normalized observation before the economizer decision. `CoolingHoldPolicy`
adds the cooling-run hold and, when enabled, the night cooling ceiling. Actuator indices come from each building's metadata,
so the controller adapts to any number of zones and AHUs.


In [ ]:
"""First baseline: average-zone control, sensor comfort bands, decoded weather.

Builds on the Apache-2.0 organizer PIDWaterResetPolicy (see vendor LICENSE).
Uses only metadata and live observations; no scenario identities or weather files.
"""
import numpy as np
from smart_buildings.baselines.pid_plus import PIDWaterResetPolicy


class BaselinePolicy(PIDWaterResetPolicy):
    USE_SENSOR_BAND = True
    # Worst-zone aggregation increased loss in both full-length pilot scenarios.
    USE_WORST_ZONE = False

    @property
    def name(self):
        return "transfer"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        stats = self.observation_normalization_metadata.get(
            "outside_air_temperature_sensor", {}
        )
        self.outside_mean = float(stats.get("sample_mean", 0.0))
        self.outside_std = float(stats.get("sample_std", 1.0))

    def _recirculation(self, obs, ctrl, u, low, high):
        # The parent's economizer expects kelvin; the observation is standardized.
        idx = ctrl.get("oat_idx")
        if idx is None:
            return 1.0
        decoded = obs.copy()
        decoded[idx] = obs[idx] * self.outside_std + self.outside_mean
        return super()._recirculation(decoded, ctrl, u, low, high)

    def get_action(self, obs):
        action = super().get_action(obs)
        # Enforce the gateway's dtype, shape and bounds contract.
        for key, dtype in (("continuous_action", np.float32),
                           ("discrete_action", np.int32)):
            space = self.action_spec[key]
            action[key] = np.clip(action[key], space.low, space.high).astype(dtype)
        return action


In [ ]:
"""Main controller: the first baseline plus a cooling-run hold for air handlers.

The organizer AHU loop stops a unit as soon as its average-zone error returns
inside the comfort band, then restarts it at the next excursion. In logs each
AHU switched 50-107 times per day. Stopping a unit that was cooling turned out
to be the baseline's main loss in hot climates. This controller keeps a unit
that was started for cooling running at its last commands for at least
HOLD_STEPS steps. Heating-season runs are unchanged.

Same behaviour as `min_on_cool_<HOLD_STEPS>` in solution/research_policy.py.
Uses only metadata and live observations; no building or weather identities.
"""
import numpy as np


class CoolingHoldPolicy(BaselinePolicy):
    HOLD_STEPS = 288
    # Outside work hours use (eco low, comfort high): no overnight drift up to
    # the eco ceiling in hot climates, full heating setback kept in cold ones.
    # Idea from the public "Neurips 2026 Smart Buildings Smartpolicy" notebook.
    NIGHT_COOL_CEILING = False

    @property
    def name(self):
        return f"cooling_hold_{self.HOLD_STEPS}"

    def _base_action(self, obs):
        if not self.NIGHT_COOL_CEILING:
            return super().get_action(obs)
        hour, day = self._get_hour_and_day(obs)
        if day < 5 and self.morning_start_hour <= hour < self.evening_start_hour:
            return super().get_action(obs)
        saved = self.eco_temp_window
        self.eco_temp_window = (saved[0], self.comfort_temp_window[1])
        try:
            return super().get_action(obs)
        finally:
            self.eco_temp_window = saved

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._on_steps = [0] * len(self.ahu_controllers)
        self._last_on = [None] * len(self.ahu_controllers)

    def get_action(self, obs):
        action = self._base_action(obs)
        cont, disc = action["continuous_action"], action["discrete_action"]
        for i, ctrl in enumerate(self.ahu_controllers):
            if not ctrl["zone_obs_indices"]:
                continue
            run = ctrl["run_discrete_idx"]
            # Supply temperature first: its sign tells cooling from heating.
            idx = [j for j in (ctrl["temp_idx"], ctrl["pressure_idx"], ctrl.get("recirc_idx"))
                   if j is not None]
            if (not disc[run] and self._on_steps[i] > 0
                    and self._on_steps[i] < self.HOLD_STEPS and self._last_on[i][0] < 0):
                disc[run] = 1
                cont[idx] = self._last_on[i]
            if disc[run]:
                self._on_steps[i] += 1
                self._last_on[i] = cont[idx].copy()
            else:
                self._on_steps[i] = 0
        return action


class ZonePIPolicy(CoolingHoldPolicy):
    """v5 controller with per-zone PI control to the band centre in work hours.

    A soft proportional zone loop stops the 2-4 K oscillation but leaves zones
    sitting steadily outside the band (no integral action). Here each zone runs
    a PI loop on (temperature - band centre) from one hour before occupancy to
    the end of the work day. One signal u in [-1, 1] drives both actuators:
    the damper opens with |u| (air is needed to cool and to carry reheat) and
    the reheat valve opens with -u. AHUs are kept running in those hours so the
    zones have air to work with. Outside them, v5 logic is unchanged.
    """
    HOLD_STEPS = 288
    NIGHT_COOL_CEILING = True
    KP = 0.2
    KI = 0.02
    PRESTART_H = 1
    # Hours before the end of the work day at which the PI loops stop (coast).
    EVENING_STOP_H = 0
    FORCED_SAT = -0.7       # normalized supply temp for a forced-on AHU (~15 C)
    FORCED_PRESSURE = -0.6  # normalized static pressure for a forced-on AHU
    KEY = "zone_pi"

    @property
    def name(self):
        return self.KEY

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._zone_integral = {}
        self._zone_u = {}
        self._pi_active = False

    def _pi_hours(self, obs):
        hour, day = self._get_hour_and_day(obs)
        return (day < 5 and self.morning_start_hour - self.PRESTART_H <= hour
                < self.evening_start_hour - self.EVENING_STOP_H)

    def _command_zones(self, continuous, zones, obs, low, high):
        if not self._pi_active:
            for vav_id, _, _ in zones:
                self._zone_integral[vav_id] = 0.0
            return super()._command_zones(continuous, zones, obs, low, high)
        lo_c, hi_c = self.comfort_temp_window
        for vav_id, obs_idx, action_indices in zones:
            if not action_indices:
                continue
            lo, hi = self._zone_band(obs, vav_id, lo_c, hi_c)
            # Before occupancy the sensor band is still the eco band; aim at comfort.
            if hi - lo > (hi_c - lo_c) + 1e-6:
                lo, hi = lo_c, hi_c
            self._cur_vav = vav_id
            err = self._denormalize_zone_temp(obs[obs_idx]) - self._zone_target(obs, lo, hi)
            integral = self._zone_integral.get(vav_id, 0.0)
            u_raw = self.KP * err + self.KI * integral
            # Conditional integration: stop winding up while saturated.
            integral *= 1.0 - self.INTEGRAL_LEAK
            if abs(u_raw) < 1.0 or np.sign(err) != np.sign(u_raw):
                limit = self.INTEGRAL_LIMIT_FRAC / self.KI
                integral = float(np.clip(integral + err, -limit, limit))
            self._zone_integral[vav_id] = integral
            u = float(np.clip(self.KP * err + self.KI * integral, -1.0, 1.0))
            if u < 0:
                u *= self.HEAT_U_SCALE
            self._max_heat = max(self._max_heat, -u)
            self._zone_u[vav_id] = u
            gain = self.HEAT_DAMPER_GAIN if u < 0 else 1.0
            damper = self.MIN_DAMPER + (1.0 - self.MIN_DAMPER) * abs(u) * gain
            reheat = max(0.0, -u)
            if self.REHEAT_FULL_AT is not None and u < 0:
                # Hottest possible discharge for the air that is moved; the
                # damper alone sets how much heat reaches the zone.
                reheat = min(1.0, -u / self.REHEAT_FULL_AT)
            split = self.HEAT_SEQUENCE_SPLIT
            if split is not None and u < 0:
                # Reheat first at minimum air; add air only once reheat is full.
                reheat = min(1.0, -u / split)
                damper = self.MIN_DAMPER + (1.0 - self.MIN_DAMPER) * max(0.0, (-u - split) / (1.0 - split))
            damper_idx = action_indices.get("supply_air_damper_percentage_command")
            if damper_idx is not None:
                continuous[damper_idx] = 2.0 * damper - 1.0
            reheat_idx = action_indices.get("reheat_valve_percentage_command")
            if reheat_idx is not None:
                continuous[reheat_idx] = 2.0 * reheat - 1.0

    MIN_DAMPER = 0.1
    # Damper opening per unit of heating demand (reheated air costs gas).
    HEAT_DAMPER_GAIN = 1.0
    # Anti-windup: integral term limited to this share of full output, and a
    # per-step leak towards zero (both neutral by default).
    INTEGRAL_LIMIT_FRAC = 1.0
    INTEGRAL_LEAK = 0.0
    # Heating sequence: None = damper and reheat both follow |u|; a number s =
    # reheat opens fully by |u| = s at minimum air, then the damper opens.
    HEAT_SEQUENCE_SPLIT = None
    # Reheat valve fully open once heating demand reaches this |u| (None =
    # valve follows |u|): each unit of cold mixed air arrives as hot as allowed.
    REHEAT_FULL_AT = None
    # Scale of the PI output when heating (softer loop for hot-discharge heating).
    HEAT_U_SCALE = 1.0
    # In PI hours, every running AHU (not only forced ones) at minimum pressure.
    PRESSURE_MIN_ALL = False

    def _zone_target(self, obs, lo, hi):
        """Temperature each zone's PI loop aims at; the band centre by default."""
        return 0.5 * (lo + hi)

    def _base_action(self, obs):
        self._pi_active = self._pi_hours(obs)
        self._max_heat = 0.0
        self._forced = []
        self._zone_u = {}
        action = super()._base_action(obs)
        if self._pi_active:
            cont, disc = action["continuous_action"], action["discrete_action"]
            # Zone reheat needs hot water even when no AHU loop asks for heat.
            if self.water_temp_idx is not None and self._max_heat > 0:
                cont[self.water_temp_idx] = max(cont[self.water_temp_idx],
                                                np.float32(2.0 * self._max_heat - 1.0))
                if self.water_pressure_idx is not None:
                    cont[self.water_pressure_idx] = 1.0
            if self.PRESSURE_MIN_ALL:
                for ctrl in self.ahu_controllers:
                    if ctrl["pressure_idx"] is not None and disc[ctrl["run_discrete_idx"]]:
                        cont[ctrl["pressure_idx"]] = -1.0
            for i, ctrl in enumerate(self.ahu_controllers):
                if ctrl["zone_obs_indices"] and not disc[ctrl["run_discrete_idx"]]:
                    self._forced.append(i)
                    disc[ctrl["run_discrete_idx"]] = 1
                    cont[ctrl["temp_idx"]] = self.FORCED_SAT
                    if ctrl["pressure_idx"] is not None:
                        cont[ctrl["pressure_idx"]] = self.FORCED_PRESSURE
            if 0 <= self.water_discrete_idx < self.n_discrete:
                disc[self.water_discrete_idx] = 1
        return action

    def get_action(self, obs):
        action = super().get_action(obs)
        # A forced-on unit is not a cooling run; do not let the hold extend it.
        for i in self._forced:
            if self._last_on[i] is not None:
                self._last_on[i][0] = 0.0
        return action


def _outside_kelvin(policy, value):
    """AHU outside-air sensors arrive in raw kelvin; decode only if normalized."""
    value = float(value)
    return value if value > 150.0 else value * policy.outside_std + policy.outside_mean


class ZonePIEconFixPolicy(ZonePIPolicy):
    """pe_lowp with a working economizer (raw-kelvin outside air temperature).

    TransferPolicy decoded the AHU outside-air sensor as if normalized, but it
    arrives in kelvin, so the economizer saw ~3000 K outside and never opened.
    """
    KP = 0.10
    KI = 0.005
    FORCED_PRESSURE = -1.0
    WARM_SAT = None
    WARM_BELOW_K = 283.15
    KEY = "econ_fix"

    def _recirculation(self, obs, ctrl, u, low, high):
        idx = ctrl.get("oat_idx")
        if idx is None:
            return 1.0
        fixed = obs.copy()
        fixed[idx] = _outside_kelvin(self, obs[idx])
        # Skip TransferPolicy's decoding; call the organizer economizer directly.
        return super(BaselinePolicy, self)._recirculation(fixed, ctrl, u, low, high)

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if self.WARM_SAT is None:
            return action
        cont = action["continuous_action"]
        for i in self._forced:
            ctrl = self.ahu_controllers[i]
            idx = ctrl.get("oat_idx")
            if idx is not None and _outside_kelvin(self, obs[idx]) < self.WARM_BELOW_K:
                cont[ctrl["temp_idx"]] = self.WARM_SAT
        return action


class DemandSatPolicy(ZonePIEconFixPolicy):
    """Supply-air temperature follows the zones' own PI demand in work hours.

    Forced-on AHUs used a fixed ~15 C supply. In cool weather that chills an
    already cool mix, then zone reheat burns gas to warm it back. Here each
    running AHU looks at the mean PI output of its zones (positive = cooling):
    with net heating demand the supply setpoint goes to maximum (no chiller);
    with cooling demand it ramps down to ~15 C, reached at mean demand RAMP
    (RAMP 0 = switch at zero demand).
    """
    RAMP = 0.0
    COLD_SAT = -0.7
    # Statistic of the zones' PI outputs that sets the supply temperature:
    # None = mean; a number = that quantile (0.75 lets hot zones dominate).
    DEMAND_QUANTILE = None
    KEY = "dsat_bin"

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if not self._pi_active:
            return action
        cont, disc = action["continuous_action"], action["discrete_action"]
        for ctrl in self.ahu_controllers:
            us = [self._zone_u[v] for v, _, _ in ctrl["zones"] if v in self._zone_u]
            if not us or not disc[ctrl["run_discrete_idx"]]:
                continue
            d = (float(np.mean(us)) if self.DEMAND_QUANTILE is None
                 else float(np.quantile(us, self.DEMAND_QUANTILE)))
            frac = (float(np.clip(d / self.RAMP, 0.0, 1.0)) if self.RAMP > 0
                    else (1.0 if d > 0 else 0.0))
            cont[ctrl["temp_idx"]] = np.float32(1.0 + frac * (self.COLD_SAT - 1.0))
        return action


class TunedPIPolicy(ZonePIEconFixPolicy):
    """econ_fix_warmcold plus switchable energy measures, all off by default.

    NIGHT_HEAT_DROP_K: lower the heating setpoint outside PI hours.
    WATER_SCALE: scale the hot-water temperature command (0..1 of range).
    TARGET_SHIFT_K: aim zones this far from the band centre towards the side
        the weather pushes them to (cheaper to hold).
    OCC_GATE: after the arrival hour, run PI hours only while the building
        reports occupants (skips holidays and empty days).
    """
    WARM_SAT = -0.4
    NIGHT_HEAT_DROP_K = 0.0
    WATER_SCALE = 1.0
    TARGET_SHIFT_K = 0.0
    OCC_GATE = False
    # Shift the target towards the low edge only in heating weather.
    COLD_SHIFT_ONLY = False
    # In PI hours, run the hot-water loop at full temperature whenever any
    # zone calls for heat (more heat per unit of cold mixed air).
    WATER_MAX_PI = False
    KEY = "tuned"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        names = list(self.field_names)
        self._occ_idx = names.index("num_occupants") if "num_occupants" in names else None
        self._oat_any = next((c["oat_idx"] for c in self.ahu_controllers
                              if c.get("oat_idx") is not None), None)

    def _pi_hours(self, obs):
        active = super()._pi_hours(obs)
        if active and self.OCC_GATE and self._occ_idx is not None:
            hour, _ = self._get_hour_and_day(obs)
            if hour >= self.morning_start_hour + 2 and float(obs[self._occ_idx]) < 0.5:
                return False
        return active

    def _zone_target(self, obs, lo, hi):
        centre = 0.5 * (lo + hi)
        if self.TARGET_SHIFT_K <= 0 or self._oat_any is None:
            return centre
        outside = _outside_kelvin(self, obs[self._oat_any])
        shift = min(self.TARGET_SHIFT_K, 0.4 * (hi - lo))
        if outside < centre:
            return centre - shift
        return centre if self.COLD_SHIFT_ONLY else centre + shift

    def _zone_band(self, obs, vav_id, low, high):
        lo, hi = super()._zone_band(obs, vav_id, low, high)
        if self.NIGHT_HEAT_DROP_K and not self._pi_active:
            lo -= self.NIGHT_HEAT_DROP_K
        return lo, hi

    def _ahu_error(self, obs, ctrl, low, high):
        if self.NIGHT_HEAT_DROP_K and not self._pi_active:
            low -= self.NIGHT_HEAT_DROP_K
        return super()._ahu_error(obs, ctrl, low, high)

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if (self.WATER_MAX_PI and self._pi_active and self.water_temp_idx is not None
                and self._max_heat > 0):
            action["continuous_action"][self.water_temp_idx] = np.float32(1.0)
        if self.WATER_SCALE != 1.0 and self.water_temp_idx is not None:
            cont = action["continuous_action"]
            frac = (float(cont[self.water_temp_idx]) + 1.0) * 0.5 * self.WATER_SCALE
            cont[self.water_temp_idx] = np.float32(2.0 * frac - 1.0)
        return action


class TunedDemandSatPolicy(TunedPIPolicy, DemandSatPolicy):
    """dsat_bin with the TunedPIPolicy switches (all off by default)."""
    WARM_SAT = None  # dsat_bin did not use the cold-weather warm supply
    KEY = "tuned_dsat"


class AdaptiveColdPolicy(TunedDemandSatPolicy):
    """Pick the heating measures from a smoothed outside temperature.

    Screening showed each cold-weather measure helps one regime and hurts
    another: an early start and full-temperature water with less heating air
    pay off only in deep cold (mean -21 C), while aiming 0.5 K below the band
    centre pays off in moderate cold but hurts deep cold. The regime follows an
    exponential average of outside air temperature over about a day.
    """
    NIGHT_HEAT_DROP_K = 2.0
    PRESTART_H = 2
    NIGHT_COOL_CEILING = False
    EXTREME_BELOW_C = -15.0
    MILD_BELOW_C = 12.0
    EXTREME_PRESTART_H = 4
    EXTREME_WATER_MAX = True
    EXTREME_HEAT_DAMPER = 0.5
    MILD_SHIFT_K = 0.5
    EMA_STEPS = 288
    # Full hot-water temperature in PI hours whenever a zone heats, any weather.
    ALWAYS_WATER_MAX = False
    KEY = "adaptive_cold"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._oat_ema = None
        self._base_prestart = self.PRESTART_H

    def _update_regime(self, obs):
        if self._oat_any is not None:
            outside = _outside_kelvin(self, obs[self._oat_any]) - 273.15
            if self._oat_ema is None:
                self._oat_ema = outside
            else:
                self._oat_ema += (outside - self._oat_ema) / self.EMA_STEPS
        ema = self._oat_ema
        extreme = ema is not None and ema < self.EXTREME_BELOW_C
        mild = ema is not None and not extreme and ema < self.MILD_BELOW_C
        self.PRESTART_H = self.EXTREME_PRESTART_H if extreme else self._base_prestart
        self.WATER_MAX_PI = bool((extreme and self.EXTREME_WATER_MAX) or self.ALWAYS_WATER_MAX)
        self.HEAT_DAMPER_GAIN = self.EXTREME_HEAT_DAMPER if extreme else 1.0
        self.TARGET_SHIFT_K = self.MILD_SHIFT_K if mild else 0.0
        self.COLD_SHIFT_ONLY = True

    def _base_action(self, obs):
        self._update_regime(obs)
        return super()._base_action(obs)


class APResearchPolicy(AdaptiveColdPolicy):
    """adapt_pre_only plus switchable hypotheses (all neutral by default).

    ZONE_SHIFT_K: per-zone economic target. Each zone keeps a ~1-day average of
        its PI output; a zone that mostly needs cooling aims up to this many K
        above its target, one that mostly needs heating aims below it.
    DYN_START_RATE: before the fixed prestart, start PI early when the mean
        zone temperature is below the comfort low edge by more than RATE K
        per hour left until occupancy (up to DYN_START_MAX_H hours ahead).
    NUDGE: negligible pressure nudge for running AHUs (noise-floor runs).
    """
    EXTREME_WATER_MAX = False
    EXTREME_HEAT_DAMPER = 1.0
    ZONE_SHIFT_K = 0.0
    ZONE_EMA_STEPS = 288
    ZONE_SHIFT_FULL_U = 0.2
    DYN_START_RATE = None
    DYN_START_MAX_H = 5
    NUDGE = 0.0
    # Idle window: in PI hours stop an AHU while every zone it serves has
    # |PI output| below IDLE_U (None = never).
    IDLE_U = None
    # Extra heating setback on weekend days (no occupancy, comfort not scored).
    WEEKEND_EXTRA_DROP_K = 0.0
    # Extra pre-start hours on Monday morning after a weekend setback.
    MONDAY_EXTRA_PRESTART_H = 0
    KEY = "ap_research"

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self._u_ema = {}
        self._all_zone_obs = [j for c in self.ahu_controllers for j in c["zone_obs_indices"]]

    def _zone_target(self, obs, lo, hi):
        target = super()._zone_target(obs, lo, hi)
        if self.ZONE_SHIFT_K <= 0:
            return target
        ema = self._u_ema.get(getattr(self, "_cur_vav", None))
        if ema is None:
            return target
        frac = float(np.clip(ema / self.ZONE_SHIFT_FULL_U, -1.0, 1.0))
        shift = min(self.ZONE_SHIFT_K, 0.4 * (hi - lo)) * frac
        return float(np.clip(target + shift, lo + 0.3, hi - 0.3))

    def _weekend_drop(self, obs):
        if not self.WEEKEND_EXTRA_DROP_K or self._pi_active:
            return 0.0
        _, day = self._get_hour_and_day(obs)
        return self.WEEKEND_EXTRA_DROP_K if day >= 5 else 0.0

    def _zone_band(self, obs, vav_id, low, high):
        lo, hi = super()._zone_band(obs, vav_id, low, high)
        return lo - self._weekend_drop(obs), hi

    def _ahu_error(self, obs, ctrl, low, high):
        return super()._ahu_error(obs, ctrl, low - self._weekend_drop(obs), high)

    def _pi_hours(self, obs):
        active = super()._pi_hours(obs)
        if not active and self.MONDAY_EXTRA_PRESTART_H:
            hour, day = self._get_hour_and_day(obs)
            if day == 0 and self.morning_start_hour - self.PRESTART_H - self.MONDAY_EXTRA_PRESTART_H <= hour < self.morning_start_hour:
                return True
        if active or self.DYN_START_RATE is None or not self._all_zone_obs:
            return active
        hour, day = self._get_hour_and_day(obs)
        left = self.morning_start_hour - hour
        if day >= 5 or left <= 0 or left > self.DYN_START_MAX_H:
            return False
        mean_t = float(np.mean([self._denormalize_zone_temp(obs[j]) for j in self._all_zone_obs]))
        return mean_t < self.comfort_temp_window[0] - self.DYN_START_RATE * left

    def _base_action(self, obs):
        action = super()._base_action(obs)
        if self.ZONE_SHIFT_K > 0:
            for v, u in self._zone_u.items():
                prev = self._u_ema.get(v, u)
                self._u_ema[v] = prev + (u - prev) / self.ZONE_EMA_STEPS
        if self.IDLE_U is not None and self._pi_active:
            cont, disc = action["continuous_action"], action["discrete_action"]
            for i, ctrl in enumerate(self.ahu_controllers):
                us = [abs(self._zone_u[v]) for v, _, _ in ctrl["zones"] if v in self._zone_u]
                if us and max(us) < self.IDLE_U and disc[ctrl["run_discrete_idx"]]:
                    disc[ctrl["run_discrete_idx"]] = 0
                    cont[ctrl["temp_idx"]] = -1.0
                    if ctrl["pressure_idx"] is not None:
                        cont[ctrl["pressure_idx"]] = -1.0
                    if i in self._forced:
                        self._forced.remove(i)
        if self.NUDGE:
            cont, disc = action["continuous_action"], action["discrete_action"]
            for ctrl in self.ahu_controllers:
                idx = ctrl["pressure_idx"]
                if idx is not None and disc[ctrl["run_discrete_idx"]]:
                    cont[idx] += np.float32(-self.NUDGE if cont[idx] > 0 else self.NUDGE)
        return action


In [ ]:
from kaggle_evaluation import smart_buildings_inference_server

class SubmissionPolicy(APResearchPolicy, smart_buildings_inference_server.Policy):
    KP = 0.1
    KI = 0.005
    HOLD_STEPS = 288
    NIGHT_COOL_CEILING = False
    ZONE_SHIFT_K = 0.6
    DEMAND_QUANTILE = 0.5
    ZONE_EMA_STEPS = 144
    WEEKEND_EXTRA_DROP_K = 6.0
    PRESSURE_MIN_ALL = True


## Inference

The official server runs the public example during a normal commit and the
hidden scenarios during a submission rerun, writing `submission.parquet`.
Internet is off.


In [ ]:
smart_buildings_inference_server.SmartBuildingsInferenceServer(SubmissionPolicy).run(
    competition_data_folder=str(DATA_ROOT),
)
